In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import json
import re

from pathlib import Path

In [3]:
BASE_DIR = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn"
)

OPENBIO_FILE = BASE_DIR / "LabeledIn_OpenBioLLM_Results.xlsx"
PUBMED_FILE = BASE_DIR / "LabeledIn_PubMedLlama_Results.xlsx"
GEMMA_FILE = BASE_DIR / "LabeledIn_Gemma3_Results.xlsx"
MEDGEMMA_FILE = BASE_DIR / "LabeledIn_MedGemma_Results.xlsx"
QWEN_FILE = BASE_DIR / "LabeledIn_Qwen3_8B_Results.xlsx"

print("OpenBio :", OPENBIO_FILE.exists())
print("PubMed  :", PUBMED_FILE.exists())
print("Gemma3  :", GEMMA_FILE.exists())
print("MedGemma:", MEDGEMMA_FILE.exists())
print("Qwen3   :", QWEN_FILE.exists())

OpenBio : True
PubMed  : True
Gemma3  : True
MedGemma: True
Qwen3   : True


In [ ]:
openbio = pd.read_excel(OPENBIO_FILE)
pubmed = pd.read_excel(PUBMED_FILE)
gemma = pd.read_excel(GEMMA_FILE)
medgemma = pd.read_excel(MEDGEMMA_FILE)
qwen = pd.read_excel(QWEN_FILE)

print("OpenBio records :", len(openbio))
print("PubMed records  :", len(pubmed))
print("Gemma3 records  :", len(gemma))
print("MedGemma records:", len(medgemma))
print("Qwen3 records   :", len(qwen))

OpenBio records : 500
PubMed records  : 500
Gemma3 records  : 500
MedGemma records: 500
Qwen3 records   : 500


In [ ]:
assert len(openbio) == 500
assert len(pubmed) == 500
assert len(gemma) == 500
assert len(medgemma) == 500
assert len(qwen) == 500

base_ids = set(openbio["record_id"])

for name, model_df in {
    "PubMed-Llama": pubmed,
    "Gemma3": gemma,
    "MedGemma": medgemma,
    "Qwen3-8B": qwen
}.items():

    same_ids = set(model_df["record_id"]) == base_ids

    print(
        name,
        "same record IDs:",
        same_ids
    )

    assert same_ids

print(
    "\n✓ All five models use the same 500 records."
)

PubMed-Llama same record IDs: True
Gemma3 same record IDs: True
MedGemma same record IDs: True
Qwen3-8B same record IDs: True

✓ All five models use the same 500 records.


In [ ]:
print("OpenBio:")
print(openbio["openbiollm_status"].value_counts(dropna=False))

print("\nPubMed:")
print(pubmed["pubmed_status"].value_counts(dropna=False))

print("\nGemma3:")
print(gemma["gemma_status"].value_counts(dropna=False))

print("\nMedGemma:")
print(medgemma["medgemma_status"].value_counts(dropna=False))

print("\nQwen3:")
print(qwen["qwen3_status"].value_counts(dropna=False))

OpenBio:
openbiollm_status
OK               470
OK_EMPTY          24
FORMAT_FAILED      5
OK_REPAIRED        1
Name: count, dtype: int64

PubMed:
pubmed_status
OK                         431
OK_REPAIRED                 57
FORMAT_FAILED_TRUNCATED     12
Name: count, dtype: int64

Gemma3:
gemma_status
OK             497
OK_REPAIRED      3
Name: count, dtype: int64

MedGemma:
medgemma_status
OK               480
FORMAT_FAILED     19
OK_EMPTY           1
Name: count, dtype: int64

Qwen3:
qwen3_status
OK          499
OK_EMPTY      1
Name: count, dtype: int64


In [ ]:
def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [ ]:
def parse_prediction(value):

    if pd.isna(value):
        return []

    text = str(value).strip()

    if text == "" or text == "[]":
        return []

    try:

        result = json.loads(text)

        if isinstance(result, list):

            cleaned = []
            seen = set()

            for item in result:

                if not isinstance(item, str):
                    continue

                item = item.strip()

                if not item:
                    continue

                norm = normalize_text(item)

                if norm not in seen:

                    seen.add(norm)
                    cleaned.append(item)

            return cleaned

    except Exception:
        pass

    return []

In [ ]:
def parse_gold_pairs(pair_string):

    """
    Convert the gold-standard CUI/mention string into:

    1. gold_cuis:
       A set containing the unique expert gold CUIs.

    2. aliases:
       A list containing:
       (normalized disease mention, CUI)

    Example:
        C0020538=Hypertension

    becomes:

        gold_cuis = {"C0020538"}

        aliases = [
            ("hypertension", "C0020538")
        ]
    """

    gold_cuis = set()
    aliases = []

    # Missing gold annotation
    if pd.isna(pair_string):
        return gold_cuis, aliases

    # Individual CUI=mention pairs are separated by |
    pieces = str(pair_string).split("|")

    for piece in pieces:

        piece = piece.strip()

        # Ignore invalid pieces
        if "=" not in piece:
            continue

        # Split only at the first =
        cui, mention = piece.split(
            "=",
            1
        )

        cui = cui.strip()
        mention = mention.strip()

        # Ignore empty values
        if not cui or not mention:
            continue

        # Store unique expert concept
        gold_cuis.add(cui)

        # Store normalized expert wording linked to its CUI
        aliases.append(
            (
                normalize_text(mention),
                cui
            )
        )

    return gold_cuis, aliases

In [ ]:
def parse_prediction(value):

    # Missing model output
    if pd.isna(value):
        return []

    text = str(value).strip()

    # Empty output
    if text == "" or text == "[]":
        return []

    try:

        # Convert JSON string to Python object
        result = json.loads(text)

        if isinstance(result, list):

            cleaned = []
            seen = set()

            for item in result:

                # Only accept text items
                if not isinstance(item, str):
                    continue

                item = item.strip()

                if not item:
                    continue

                # Normalize only for duplicate detection
                norm = normalize_text(item)

                # Avoid duplicate predictions
                if norm not in seen:

                    seen.add(norm)
                    cleaned.append(item)

            return cleaned

    except Exception:
        pass

    # Invalid/unusable prediction
    return []

In [ ]:
def match_prediction_to_gold(
    predictions,
    gold_cuis,
    aliases
):

    # Gold CUIs successfully recovered by the model
    matched_cuis = set()

    # Model predictions that cannot be matched
    # to an expert gold concept
    unmatched_predictions = []

    # --------------------------------------------------
    # Sort expert aliases from longest to shortest
    #
    # This gives more specific disease phrases priority.
    # --------------------------------------------------

    aliases_sorted = sorted(
        aliases,
        key=lambda x: len(
            x[0].split()
        ),
        reverse=True
    )

    # --------------------------------------------------
    # Evaluate every model prediction
    # --------------------------------------------------

    for prediction in predictions:

        pred_norm = normalize_text(
            prediction
        )

        if not pred_norm:
            continue

        matched_cui = None

        # ==================================================
        # STEP 1:
        # Exact normalized match
        # ==================================================

        for alias, cui in aliases_sorted:

            if pred_norm == alias:

                matched_cui = cui
                break

        # ==================================================
        # STEP 2:
        # Conservative multi-word containment
        #
        # Example:
        #
        # Expert:
        #   ventricular tachycardia
        #
        # Model:
        #   hemodynamically unstable ventricular tachycardia
        #
        # This can be accepted.
        #
        # But containment is NOT allowed for single-word
        # expert terms because this could create many
        # false matches.
        # ==================================================

        if matched_cui is None:

            for alias, cui in aliases_sorted:

                alias_words = alias.split()

                # Do not use containment for one-word aliases
                if len(alias_words) < 2:
                    continue

                if (
                    alias in pred_norm
                    or
                    pred_norm in alias
                ):

                    matched_cui = cui
                    break

        # ==================================================
        # Store matching result
        # ==================================================

        if matched_cui is not None:

            matched_cuis.add(
                matched_cui
            )

        else:

            unmatched_predictions.append(
                prediction
            )

    # ======================================================
    # Calculate TP, FP and FN
    # ======================================================

    # True Positive:
    # expert gold concepts correctly recovered
    tp = len(
        matched_cuis.intersection(
            gold_cuis
        )
    )

    # False Negative:
    # expert gold concepts missed by the model
    fn = len(
        gold_cuis
        -
        matched_cuis
    )

    # False Positive:
    # model predictions that could not be matched
    # to any expert gold concept
    fp = len(
        unmatched_predictions
    )

    # ======================================================
    # Return detailed results
    # ======================================================

    return {

        "TP": tp,

        "FP": fp,

        "FN": fn,

        "matched_cuis":
            sorted(
                matched_cuis
            ),

        "unmatched_predictions":
            unmatched_predictions,

        "missed_cuis":
            sorted(
                gold_cuis
                -
                matched_cuis
            )
    }

In [ ]:
def evaluate_model(
    df,
    output_column,
    status_column,
    model_name
):

    rows = []

    for _, row in df.iterrows():

        record_id = row["record_id"]

        # --------------------------------------------
        # Read expert gold concepts
        # --------------------------------------------
        gold_cuis, aliases = parse_gold_pairs(
            row["gold_cui_mention_pairs"]
        )

        # --------------------------------------------
        # Read model status
        # --------------------------------------------
        status = str(
            row.get(status_column, "")
        ).strip()

        # --------------------------------------------
        # If the model output was unusable,
        # treat it as producing no prediction
        # --------------------------------------------
        if status in [
            "FORMAT_FAILED",
            "FORMAT_FAILED_TRUNCATED",
            "TRUNCATED",
            "FAILED"
        ]:

            predictions = []

        else:

            predictions = parse_prediction(
                row.get(
                    output_column,
                    ""
                )
            )

        # --------------------------------------------
        # Compare model predictions against gold
        # --------------------------------------------
        result = match_prediction_to_gold(
            predictions,
            gold_cuis,
            aliases
        )

        # --------------------------------------------
        # Store record-level results
        # --------------------------------------------
        rows.append({

            "record_id":
                record_id,

            "model":
                model_name,

            "status":
                status,

            "gold_cui_count":
                len(gold_cuis),

            "prediction_count":
                len(predictions),

            "TP":
                result["TP"],

            "FP":
                result["FP"],

            "FN":
                result["FN"],

            "predictions":
                json.dumps(
                    predictions,
                    ensure_ascii=False
                ),

            "matched_cuis":
                "; ".join(
                    result["matched_cuis"]
                ),

            "missed_cuis":
                "; ".join(
                    result["missed_cuis"]
                ),

            "unmatched_predictions":
                json.dumps(
                    result[
                        "unmatched_predictions"
                    ],
                    ensure_ascii=False
                )
        })

    # --------------------------------------------
    # Convert all record results to DataFrame
    # --------------------------------------------
    detail = pd.DataFrame(rows)

    # --------------------------------------------
    # Aggregate TP, FP, FN
    # --------------------------------------------
    TP = detail["TP"].sum()
    FP = detail["FP"].sum()
    FN = detail["FN"].sum()

    # --------------------------------------------
    # Precision
    # --------------------------------------------
    precision = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else 0
    )

    # --------------------------------------------
    # Recall
    # --------------------------------------------
    recall = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else 0
    )

    # --------------------------------------------
    # F1-score
    # --------------------------------------------
    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    # --------------------------------------------
    # Model summary
    # --------------------------------------------
    summary = {

        "Model":
            model_name,

        "Records":
            len(detail),

        "TP":
            int(TP),

        "FP":
            int(FP),

        "FN":
            int(FN),

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1
    }

    return summary, detail

In [ ]:
openbio_summary, openbio_detail = evaluate_model(
    openbio,
    output_column="openbiollm_output",
    status_column="openbiollm_status",
    model_name="OpenBioLLM"
)

openbio_summary

{'Model': 'OpenBioLLM',
 'Records': 500,
 'TP': 848,
 'FP': 371,
 'FN': 891,
 'Precision': np.float64(0.6956521739130435),
 'Recall': np.float64(0.48763657274295574),
 'F1': np.float64(0.5733603786342124)}

In [ ]:
pubmed_summary, pubmed_detail = evaluate_model(
    pubmed,
    output_column="pubmed_output",
    status_column="pubmed_status",
    model_name="PubMed-Llama"
)

pubmed_summary

{'Model': 'PubMed-Llama',
 'Records': 500,
 'TP': 1131,
 'FP': 818,
 'FN': 608,
 'Precision': np.float64(0.5802975885069266),
 'Recall': np.float64(0.6503737780333525),
 'F1': np.float64(0.6133405639913232)}

In [ ]:
gemma_summary, gemma_detail = evaluate_model(
    gemma,
    output_column="gemma_output",
    status_column="gemma_status",
    model_name="Gemma3"
)

gemma_summary

{'Model': 'Gemma3',
 'Records': 500,
 'TP': 1317,
 'FP': 982,
 'FN': 422,
 'Precision': np.float64(0.5728577642453241),
 'Recall': np.float64(0.7573317998849913),
 'F1': np.float64(0.6523031203566122)}

In [ ]:
medgemma_summary, medgemma_detail = evaluate_model(
    medgemma,
    output_column="medgemma_output",
    status_column="medgemma_status",
    model_name="MedGemma"
)

medgemma_summary

{'Model': 'MedGemma',
 'Records': 500,
 'TP': 1185,
 'FP': 1022,
 'FN': 554,
 'Precision': np.float64(0.536927956502039),
 'Recall': np.float64(0.6814261069580219),
 'F1': np.float64(0.6006082108464267)}

In [ ]:
qwen_summary, qwen_detail = evaluate_model(
    qwen,
    output_column="qwen3_output",
    status_column="qwen3_status",
    model_name="Qwen3-8B"
)

qwen_summary

{'Model': 'Qwen3-8B',
 'Records': 500,
 'TP': 1084,
 'FP': 539,
 'FN': 655,
 'Precision': np.float64(0.6678989525569933),
 'Recall': np.float64(0.6233467510063255),
 'F1': np.float64(0.6448542534205831)}

In [ ]:
summary_df = pd.DataFrame(
    [
        openbio_summary,
        pubmed_summary,
        gemma_summary,
        medgemma_summary,
        qwen_summary
    ]
)

summary_df[
    ["Precision", "Recall", "F1"]
] = summary_df[
    ["Precision", "Recall", "F1"]
].round(4)

display(summary_df)

,Model,Records,TP,FP,FN,Precision,Recall,F1
0,OpenBioLLM,500,848,371,891,0.6957,0.4876,0.5734
1,PubMed-Llama,500,1131,818,608,0.5803,0.6504,0.6133
2,Gemma3,500,1317,982,422,0.5729,0.7573,0.6523
3,MedGemma,500,1185,1022,554,0.5369,0.6814,0.6006
4,Qwen3-8B,500,1084,539,655,0.6679,0.6233,0.6449


In [ ]:
summary_df["Gold_Concepts"] = (
    summary_df["TP"]
    +
    summary_df["FN"]
)

display(
    summary_df[
        [
            "Model",
            "Records",
            "TP",
            "FP",
            "FN",
            "Gold_Concepts",
            "Precision",
            "Recall",
            "F1"
        ]
    ]
)

,Model,Records,TP,FP,FN,Gold_Concepts,Precision,Recall,F1
0,OpenBioLLM,500,848,371,891,1739,0.6957,0.4876,0.5734
1,PubMed-Llama,500,1131,818,608,1739,0.5803,0.6504,0.6133
2,Gemma3,500,1317,982,422,1739,0.5729,0.7573,0.6523
3,MedGemma,500,1185,1022,554,1739,0.5369,0.6814,0.6006
4,Qwen3-8B,500,1084,539,655,1739,0.6679,0.6233,0.6449


In [ ]:
openbio_detail
pubmed_detail
gemma_detail
medgemma_detail
qwen_detail

,record_id,model,status,gold_cui_count,prediction_count,TP,FP,FN,predictions,matched_cuis,missed_cuis,unmatched_predictions
0,10004,Qwen3-8B,OK,3,4,3,0,0,"[""Erosive Esophagitis Associated With Gastroes...",C0017168; C0043515; C0267055,,[]
1,10008,Qwen3-8B,OK,4,2,2,0,2,"[""anxiety disorder"", ""Panic Disorder""]",C0003469; C0030319,C0001818; C0338908,[]
2,10012,Qwen3-8B,OK,2,3,2,1,0,"[""ventricular fibrillation"", ""hemodynamically ...",C0042510; C0042514,,"[""VT VF""]"
3,10016,Qwen3-8B,OK,2,2,1,1,1,"[""inflammatory and pruritic manifestations of ...",C0406317,C0037274,"[""inflammatory and pruritic manifestations of ..."
4,10017,Qwen3-8B,OK,1,1,1,0,0,"[""acute, painful musculoskeletal conditions""]",C0184567,,[]
...,...,...,...,...,...,...,...,...,...,...,...,...
495,2817,Qwen3-8B,OK,3,3,2,1,1,"[""Pneumocystis carinii pneumonia"", ""enteritis ...",C0032305; C0042029,C0013371,"[""enteritis caused by susceptible strains of S..."
496,3091,Qwen3-8B,OK,3,3,2,1,1,"[""Osteoarthritis ( OA )"", ""Rheumatoid Arthriti...",C0003873; C0553662,C0029408,"[""Osteoarthritis ( OA )""]"
497,4060,Qwen3-8B,OK,1,1,0,1,1,"[""gout flares""]",,C0018099,"[""gout flares""]"
498,5926,Qwen3-8B,OK,1,2,1,1,0,"[""mydriasis and cycloplegia in diagnostic proc...",C0022073,,"[""mydriasis and cycloplegia in diagnostic proc..."


In [ ]:
import numpy as np
import pandas as pd

def calculate_micro_metrics(detail_df):

    TP = detail_df["TP"].sum()
    FP = detail_df["FP"].sum()
    FN = detail_df["FN"].sum()

    precision = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else 0
    )

    recall = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return precision, recall, f1

In [ ]:
def bootstrap_ci(
    detail_df,
    n_bootstrap=10000,
    seed=42
):

    rng = np.random.default_rng(seed)

    n = len(detail_df)

    precision_values = []
    recall_values = []
    f1_values = []

    for _ in range(n_bootstrap):

        sample_indices = rng.integers(
            0,
            n,
            size=n
        )

        sample = detail_df.iloc[
            sample_indices
        ]

        precision, recall, f1 = (
            calculate_micro_metrics(sample)
        )

        precision_values.append(
            precision
        )

        recall_values.append(
            recall
        )

        f1_values.append(
            f1
        )

    result = {
        "Precision_Lower": np.percentile(
            precision_values,
            2.5
        ),

        "Precision_Upper": np.percentile(
            precision_values,
            97.5
        ),

        "Recall_Lower": np.percentile(
            recall_values,
            2.5
        ),

        "Recall_Upper": np.percentile(
            recall_values,
            97.5
        ),

        "F1_Lower": np.percentile(
            f1_values,
            2.5
        ),

        "F1_Upper": np.percentile(
            f1_values,
            97.5
        )
    }

    return result

In [ ]:
models_detail = {
    "OpenBioLLM": openbio_detail,
    "PubMed-Llama": pubmed_detail,
    "Gemma3": gemma_detail,
    "MedGemma": medgemma_detail,
    "Qwen3-8B": qwen_detail
}

ci_rows = []

for model_name, detail_df in models_detail.items():

    precision, recall, f1 = (
        calculate_micro_metrics(
            detail_df
        )
    )

    ci = bootstrap_ci(
        detail_df,
        n_bootstrap=10000,
        seed=42
    )

    ci_rows.append(
        {
            "Model": model_name,

            "Precision":
                precision,

            "Precision_95CI_Lower":
                ci["Precision_Lower"],

            "Precision_95CI_Upper":
                ci["Precision_Upper"],

            "Recall":
                recall,

            "Recall_95CI_Lower":
                ci["Recall_Lower"],

            "Recall_95CI_Upper":
                ci["Recall_Upper"],

            "F1":
                f1,

            "F1_95CI_Lower":
                ci["F1_Lower"],

            "F1_95CI_Upper":
                ci["F1_Upper"]
        }
    )

ci_df = pd.DataFrame(
    ci_rows
)

ci_df = ci_df.round(4)

display(ci_df)

,Model,Precision,Precision_95CI_Lower,Precision_95CI_Upper,Recall,Recall_95CI_Lower,Recall_95CI_Upper,F1,F1_95CI_Lower,F1_95CI_Upper
0,OpenBioLLM,0.6957,0.6465,0.7457,0.4876,0.4304,0.5435,0.5734,0.5258,0.6204
1,PubMed-Llama,0.5803,0.5394,0.6222,0.6504,0.5953,0.6983,0.6133,0.5781,0.6476
2,Gemma3,0.5729,0.5359,0.6104,0.7573,0.7358,0.7790,0.6523,0.6258,0.6785
3,MedGemma,0.5369,0.4971,0.5769,0.6814,0.6431,0.7187,0.6006,0.5704,0.6299
4,Qwen3-8B,0.6679,0.6298,0.7057,0.6233,0.5809,0.6580,0.6449,0.6135,0.6746


In [ ]:
base_ids = gemma_detail[
    "record_id"
].tolist()

for model_name, detail_df in models_detail.items():

    assert (
        detail_df["record_id"].tolist()
        ==
        base_ids
    ), (
        f"Record order mismatch: "
        f"{model_name}"
    )

print(
    "✓ All five detail tables have "
    "the same record order."
)

✓ All five detail tables have the same record order.


In [ ]:
def paired_bootstrap_difference(
    detail_a,
    detail_b,
    metric="F1",
    n_bootstrap=10000,
    seed=42
):

    assert (
        detail_a["record_id"].tolist()
        ==
        detail_b["record_id"].tolist()
    )

    rng = np.random.default_rng(seed)

    n = len(detail_a)

    differences = []

    for _ in range(n_bootstrap):

        sample_indices = rng.integers(
            0,
            n,
            size=n
        )

        sample_a = detail_a.iloc[
            sample_indices
        ]

        sample_b = detail_b.iloc[
            sample_indices
        ]

        p_a, r_a, f1_a = (
            calculate_micro_metrics(
                sample_a
            )
        )

        p_b, r_b, f1_b = (
            calculate_micro_metrics(
                sample_b
            )
        )

        metrics_a = {
            "Precision": p_a,
            "Recall": r_a,
            "F1": f1_a
        }

        metrics_b = {
            "Precision": p_b,
            "Recall": r_b,
            "F1": f1_b
        }

        difference = (
            metrics_a[metric]
            -
            metrics_b[metric]
        )

        differences.append(
            difference
        )

    differences = np.array(
        differences
    )

    lower = np.percentile(
        differences,
        2.5
    )

    upper = np.percentile(
        differences,
        97.5
    )

    mean_difference = (
        differences.mean()
    )

    # Approximate two-sided bootstrap p-value
    p_value = (
        2
        *
        min(
            np.mean(
                differences <= 0
            ),
            np.mean(
                differences >= 0
            )
        )
    )

    p_value = min(
        p_value,
        1.0
    )

    return {
        "Mean_Difference":
            mean_difference,

        "CI_Lower":
            lower,

        "CI_Upper":
            upper,

        "P_Value":
            p_value
    }

In [ ]:
gemma_vs_qwen = paired_bootstrap_difference(
    gemma_detail,
    qwen_detail,
    metric="F1",
    n_bootstrap=10000,
    seed=42
)

gemma_vs_qwen

{'Mean_Difference': np.float64(0.007485151847469108),
 'CI_Lower': np.float64(-0.027178886339232877),
 'CI_Upper': np.float64(0.04461958759295979),
 'P_Value': np.float64(0.693)}

In [ ]:
comparisons = {
    "Gemma3 vs OpenBioLLM":
        openbio_detail,

    "Gemma3 vs PubMed-Llama":
        pubmed_detail,

    "Gemma3 vs MedGemma":
        medgemma_detail,

    "Gemma3 vs Qwen3-8B":
        qwen_detail
}

comparison_rows = []

for comparison_name, other_detail in comparisons.items():

    result = paired_bootstrap_difference(
        gemma_detail,
        other_detail,
        metric="F1",
        n_bootstrap=10000,
        seed=42
    )

    comparison_rows.append(
        {
            "Comparison":
                comparison_name,

            "F1_Difference":
                result[
                    "Mean_Difference"
                ],

            "95CI_Lower":
                result[
                    "CI_Lower"
                ],

            "95CI_Upper":
                result[
                    "CI_Upper"
                ],

            "P_Value":
                result[
                    "P_Value"
                ]
        }
    )

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df = comparison_df.round(
    4
)

display(
    comparison_df
)

,Comparison,F1_Difference,95CI_Lower,95CI_Upper,P_Value
0,Gemma3 vs OpenBioLLM,0.0786,0.0257,0.1344,0.0018
1,Gemma3 vs PubMed-Llama,0.0391,0.0069,0.0744,0.0158
2,Gemma3 vs MedGemma,0.0519,0.0243,0.0810,0.0000
3,Gemma3 vs Qwen3-8B,0.0075,-0.0272,0.0446,0.6930


In [ ]:
metrics = [
    "Precision",
    "Recall",
    "F1"
]

gemma_qwen_rows = []

for metric in metrics:

    result = paired_bootstrap_difference(
        gemma_detail,
        qwen_detail,
        metric=metric,
        n_bootstrap=10000,
        seed=42
    )

    gemma_qwen_rows.append(
        {
            "Metric":
                metric,

            "Gemma_minus_Qwen":
                result[
                    "Mean_Difference"
                ],

            "95CI_Lower":
                result[
                    "CI_Lower"
                ],

            "95CI_Upper":
                result[
                    "CI_Upper"
                ],

            "P_Value":
                result[
                    "P_Value"
                ]
        }
    )

gemma_qwen_df = pd.DataFrame(
    gemma_qwen_rows
).round(4)

display(
    gemma_qwen_df
)

,Metric,Gemma_minus_Qwen,95CI_Lower,95CI_Upper,P_Value
0,Precision,-0.0951,-0.1369,-0.0558,0.000
1,Recall,0.1339,0.0975,0.1759,0.000
2,F1,0.0075,-0.0272,0.0446,0.693


In [ ]:
BOOTSTRAP_FILE = (
    BASE_DIR
    / "LabeledIn_5_Model_Bootstrap_Statistics.xlsx"
)

with pd.ExcelWriter(
    BOOTSTRAP_FILE,
    engine="openpyxl"
) as writer:

    ci_df.to_excel(
        writer,
        sheet_name="Bootstrap_95CI",
        index=False
    )

    comparison_df.to_excel(
        writer,
        sheet_name="Gemma_F1_Comparisons",
        index=False
    )

    gemma_qwen_df.to_excel(
        writer,
        sheet_name="Gemma_vs_Qwen",
        index=False
    )

print(
    "Saved:"
)

print(
    BOOTSTRAP_FILE
)

Saved:
/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_5_Model_Bootstrap_Statistics.xlsx
